# Módulo 6 · Clase 1 — Vectores con NumPy

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | ¿Por qué matemáticas para IA? |
| 2 | NumPy — la calculadora del módulo |
| 3 | Vectores: qué son y cómo representarlos |
| 4 | Magnitud y dirección |
| 5 | Operaciones con vectores |
| 6 | Producto punto |
| 7 | Similitud coseno — aplicación real en IA |

> **Nota:** NumPy es la librería base de todo el módulo. La usamos desde esta clase en adelante.

In [ ]:
import subprocess, sys
for pkg in ['numpy', 'matplotlib']:
    try:
        __import__(pkg)
        print(f"{pkg}: ya instalado")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: instalado")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"NumPy versión: {np.__version__}")

---
## 1. ¿Por qué matemáticas para IA?

```
Un modelo de lenguaje como GPT representa cada palabra como un VECTOR
de cientos o miles de números.

"gato"  → [0.2,  0.8, -0.1,  0.5, ...]
"perro" → [0.3,  0.7, -0.2,  0.4, ...]
"coche" → [-0.5, 0.1,  0.9,  0.0, ...]

Para saber si dos palabras son similares, calculamos
la similitud coseno entre sus vectores.

Para entrenar una red neuronal, multiplicamos matrices
millones de veces por segundo.

Este módulo te da las herramientas matemáticas para entender
por dentro cómo funciona la IA.
```

---
## 2. NumPy

NumPy (Numerical Python) es la base de todo el ecosistema científico de Python:

```
NumPy
  ├── pandas   (análisis de datos)
  ├── scikit-learn (machine learning)
  ├── TensorFlow / PyTorch (deep learning)
  └── matplotlib (visualización)

Ventajas frente a listas Python:
  ✓ 10-100× más rápido (operaciones en C)
  ✓ Operaciones vectorizadas (sin bucles)
  ✓ Broadcasting (operaciones entre arrays de distinto tamaño)
  ✓ Funciones matemáticas integradas
```

In [ ]:
# ── Crear arrays ─────────────────────────────────────────────
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("Array a:", a)
print("Tipo:   ", type(a))
print("dtype:  ", a.dtype)
print("shape:  ", a.shape)

print("\n── Operaciones vectorizadas (sin bucles) ──")
print("a + b  =", a + b)      # suma elemento a elemento
print("a * 3  =", a * 3)      # escalar
print("a ** 2 =", a ** 2)     # potencia
print("a > 3  =", a > 3)      # comparación booleana

print("\n── vs listas Python (requieren bucles) ──")
lista = [1, 2, 3, 4, 5]
# lista * 3 → repite la lista, no multiplica cada elemento
print("lista * 3 =", lista * 3)   # ← NO es lo que queremos en matemáticas
print("array * 3 =", a * 3)       # ← SÍ es correcto

---
## 3. ¿Qué es un vector?

```
Un vector es una lista ordenada de números.

En 2D:  v = [3, 4]   → punto en el plano, o flecha desde el origen
En 3D:  v = [1, 2, 3] → punto en el espacio
En IA:  v = [0.2, -0.5, 0.8, ..., 0.1]  → 768 dimensiones (BERT)

Cada número es una COORDENADA o COMPONENTE del vector.

      ↑  y
   4  │      ●  (3, 4)
   3  │    ↗
   2  │  ↗
   1  │↗
      └──────→ x
         1  2  3
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Izquierda: vectores en 2D
ax = axes[0]
vectores = {
    "v = [3, 4]": ([3, 4], "blue"),
    "u = [-2, 3]": ([-2, 3], "red"),
    "w = [4, -1]": ([4, -1], "green"),
}
for nombre, (vec, color) in vectores.items():
    ax.annotate("", xy=(vec[0], vec[1]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.text(vec[0]*0.5, vec[1]*0.5, nombre, color=color, fontsize=9)

ax.set_xlim(-3, 5); ax.set_ylim(-2, 5)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.grid(True, alpha=0.3); ax.set_title("Vectores en 2D")
ax.set_xlabel("x"); ax.set_ylabel("y")

# Derecha: suma de vectores
ax2 = axes[1]
v = np.array([3, 1])
u = np.array([1, 2])
suma = v + u

ax2.annotate("", xy=v, xytext=(0,0), arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax2.annotate("", xy=suma, xytext=v, arrowprops=dict(arrowstyle="->", color="red", lw=2, linestyle="dashed"))
ax2.annotate("", xy=suma, xytext=(0,0), arrowprops=dict(arrowstyle="->", color="purple", lw=2.5))
ax2.text(v[0]/2, v[1]/2-0.2, "v=[3,1]", color="blue", fontsize=9)
ax2.text(v[0]+0.1, v[1]+0.5, "u=[1,2]", color="red", fontsize=9)
ax2.text(suma[0]/2-0.5, suma[1]/2+0.2, f"v+u={list(suma)}", color="purple", fontsize=9, fontweight="bold")

ax2.set_xlim(-1, 6); ax2.set_ylim(-1, 4)
ax2.axhline(0, color='gray', lw=0.5); ax2.axvline(0, color='gray', lw=0.5)
ax2.grid(True, alpha=0.3); ax2.set_title("Suma de vectores")
ax2.set_xlabel("x"); ax2.set_ylabel("y")

plt.tight_layout()
plt.show()

---
## 4. Magnitud y dirección

```
MAGNITUD (módulo, norma) = longitud del vector

  Para v = [3, 4]:
  ‖v‖ = √(3² + 4²) = √(9 + 16) = √25 = 5

  En general:
  ‖v‖ = √(v₁² + v₂² + ... + vₙ²)

DIRECCIÓN = vector unitario (magnitud = 1)

  v̂ = v / ‖v‖

  Para v = [3, 4]:
  v̂ = [3/5, 4/5] = [0.6, 0.8]
  ‖v̂‖ = √(0.6² + 0.8²) = √(0.36 + 0.64) = 1  ✓

¿Para qué sirve el vector unitario?
→ Separa la longitud (cuánto) de la dirección (hacia dónde)
→ En IA: la similitud coseno usa vectores unitarios
```

In [ ]:
v = np.array([3, 4])

# Magnitud manual
magnitud_manual = np.sqrt(np.sum(v ** 2))

# Magnitud con NumPy
magnitud_numpy = np.linalg.norm(v)

print(f"Vector:           {v}")
print(f"Magnitud (manual): {magnitud_manual}")
print(f"Magnitud (numpy):  {magnitud_numpy}")

# Vector unitario (normalizar)
v_unitario = v / np.linalg.norm(v)
print(f"\nVector unitario:  {v_unitario}")
print(f"‖v̂‖ = {np.linalg.norm(v_unitario):.6f}  (debe ser 1)")

# Varios vectores con distinta magnitud, misma dirección
print("\n── Misma dirección, distinta magnitud ──")
for escalar in [0.5, 1, 2, 5]:
    w = v * escalar
    w_u = w / np.linalg.norm(w)
    print(f"  {escalar}×v = {w}  → ‖w‖={np.linalg.norm(w):.1f}  → v̂={np.round(w_u, 4)}")

print("→ El vector unitario siempre es el mismo aunque la magnitud cambie")

---
## 5. Operaciones con vectores

In [ ]:
v = np.array([1, 2, 3])
u = np.array([4, 5, 6])

print("=== Operaciones básicas ===")
print(f"v = {v}")
print(f"u = {u}")
print(f"v + u  = {v + u}     ← suma elemento a elemento")
print(f"v - u  = {v - u}     ← resta")
print(f"3 × v  = {3 * v}     ← producto por escalar")
print(f"v × u  = {v * u}    ← producto elemento a elemento (NO producto punto)")

print("\n=== Propiedades ===")
print(f"v + u == u + v:   {np.array_equal(v + u, u + v)}  (conmutativa)")
w = np.array([1, 0, -1])
print(f"(v+u)+w == v+(u+w): {np.array_equal((v+u)+w, v+(u+w))}  (asociativa)")

print("\n=== Funciones de agregación ===")
print(f"np.sum(v)  = {np.sum(v)}")
print(f"np.mean(v) = {np.mean(v)}")
print(f"np.max(v)  = {np.max(v)}")
print(f"np.min(v)  = {np.min(v)}")

---
## 6. Producto punto (dot product)

```
v · u = v₁×u₁ + v₂×u₂ + ... + vₙ×uₙ

También: v · u = ‖v‖ × ‖u‖ × cos(θ)

Donde θ es el ángulo entre los vectores.

Interpretación:
  v · u > 0  → vectores apuntan en dirección similar   (θ < 90°)
  v · u = 0  → vectores perpendiculares               (θ = 90°)
  v · u < 0  → vectores apuntan en dirección opuesta  (θ > 90°)

En IA: se usa constantemente en redes neuronales (cada neurona
calcula el producto punto entre sus pesos y la entrada).
```

In [ ]:
v = np.array([1, 2, 3])
u = np.array([4, 5, 6])

# Tres formas de calcular el producto punto
dot_manual = sum(vi * ui for vi, ui in zip(v, u))
dot_numpy1 = np.dot(v, u)
dot_numpy2 = v @ u          # operador @ (Python 3.5+)

print(f"v = {v}")
print(f"u = {u}")
print(f"v · u (manual) = {1*4} + {2*5} + {3*6} = {dot_manual}")
print(f"v · u (np.dot) = {dot_numpy1}")
print(f"v · u (@ oper) = {dot_numpy2}")

# Verificar con la fórmula del ángulo
cos_theta = dot_numpy1 / (np.linalg.norm(v) * np.linalg.norm(u))
theta_rad = np.arccos(cos_theta)
theta_grados = np.degrees(theta_rad)
print(f"\nÁngulo entre v y u: {theta_grados:.2f}°")

# Casos especiales
print("\n── Casos especiales ──")
a = np.array([1, 0])
b_paralelo    = np.array([3, 0])  # misma dirección
b_perpendic   = np.array([0, 2])  # perpendicular
b_opuesto     = np.array([-2, 0]) # dirección opuesta

for nombre, b in [("paralelo", b_paralelo), ("perpendicular", b_perpendic), ("opuesto", b_opuesto)]:
    print(f"  a={a} · b_{nombre}={b}: dot={np.dot(a,b)}, ángulo={np.degrees(np.arccos(np.clip(np.dot(a/np.linalg.norm(a), b/np.linalg.norm(b)), -1, 1))):.0f}°")

---
## 7. Similitud coseno — aplicación real en IA

```
similitud_coseno(v, u) = (v · u) / (‖v‖ × ‖u‖)

Resultado entre -1 y 1:
   1   → vectores idénticos en dirección
   0   → vectores perpendiculares (sin relación)
  -1   → vectores opuestos

Ventaja sobre distancia euclidiana:
→ La magnitud no importa, solo la dirección
→ "gato" × 1000 es igual de similar a "perro" que "gato" × 1

Uso en IA:
→ Búsqueda semántica (encontrar textos similares)
→ Sistemas de recomendación
→ Detección de duplicados
```

In [ ]:
def similitud_coseno(v, u):
    return np.dot(v, u) / (np.linalg.norm(v) * np.linalg.norm(u))


# Simulamos embeddings de palabras (en la realidad son vectores de 768+ dims)
# Usamos 4 dimensiones para que sea comprensible
# Dimensiones: [animal, doméstico, mecánico, veloz]
embeddings = {
    "gato":     np.array([0.9,  0.8, -0.1,  0.3]),
    "perro":    np.array([0.8,  0.9, -0.1,  0.5]),
    "tigre":    np.array([0.9, -0.3, -0.1,  0.7]),
    "coche":    np.array([-0.1, -0.2,  0.9,  0.8]),
    "bicicleta":np.array([-0.1, -0.1,  0.7,  0.4]),
    "avión":    np.array([-0.1, -0.2,  0.8,  0.9]),
}

print("Similitudes coseno entre palabras:")
print(f"{'Par':<25} {'Similitud':>10} {'Interpretación'}")
print("-" * 60)

pares = [
    ("gato",      "perro"),
    ("gato",      "tigre"),
    ("gato",      "coche"),
    ("coche",     "bicicleta"),
    ("coche",     "avión"),
    ("gato",      "gato"),
]
for p1, p2 in pares:
    sim = similitud_coseno(embeddings[p1], embeddings[p2])
    if sim > 0.9:
        interp = "muy similares"
    elif sim > 0.6:
        interp = "relacionadas"
    elif sim > 0.2:
        interp = "algo relacionadas"
    else:
        interp = "sin relación"
    print(f"  {p1} ↔ {p2:<12} {sim:>8.4f}   {interp}")

# Visualización del mapa de similitudes
palabras = list(embeddings.keys())
n = len(palabras)
matriz_sim = np.zeros((n, n))
for i, p1 in enumerate(palabras):
    for j, p2 in enumerate(palabras):
        matriz_sim[i, j] = similitud_coseno(embeddings[p1], embeddings[p2])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matriz_sim, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(palabras, rotation=45, ha="right")
ax.set_yticklabels(palabras)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{matriz_sim[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="Similitud coseno")
ax.set_title("Mapa de similitud entre palabras")
plt.tight_layout()
plt.show()

---
## Práctica

### Ejercicio 1 — Propiedades de vectores

**Enunciado:** Dado `v = [3, -4, 0, 5]` y `u = [1, 2, -3, 4]`:
1. Calcula la magnitud de `v` y de `u`
2. Obtén los vectores unitarios de `v` y `u`
3. Verifica que `‖v̂‖ = 1` para ambos
4. Calcula `2v + 3u` y su magnitud
5. ¿Cuál es el ángulo en grados entre `v` y `u`?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import numpy as np
v = np.array([3, -4, 0, 5])
u = np.array([1, 2, -3, 4])


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
v = np.array([3, -4, 0, 5])
u = np.array([1, 2, -3, 4])

# 1. Magnitudes
mag_v = np.linalg.norm(v)
mag_u = np.linalg.norm(u)
print(f"‖v‖ = {mag_v:.4f}")
print(f"‖u‖ = {mag_u:.4f}")

# 2. Vectores unitarios
v_hat = v / mag_v
u_hat = u / mag_u
print(f"\nv̂ = {np.round(v_hat, 4)}")
print(f"û = {np.round(u_hat, 4)}")

# 3. Verificar magnitud = 1
print(f"\n‖v̂‖ = {np.linalg.norm(v_hat):.6f}")
print(f"‖û‖ = {np.linalg.norm(u_hat):.6f}")

# 4. Combinación lineal
combo = 2*v + 3*u
print(f"\n2v + 3u = {combo}")
print(f"‖2v + 3u‖ = {np.linalg.norm(combo):.4f}")

# 5. Ángulo
cos_theta = np.dot(v, u) / (mag_v * mag_u)
theta = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))
print(f"\nÁngulo entre v y u: {theta:.2f}°")
```
</details>

---
### Ejercicio 2 — Producto punto

**Enunciado:** En una red neuronal, una neurona calcula la suma ponderada de sus entradas:

`salida = w · x + b`

Donde `w` son los pesos, `x` la entrada y `b` el sesgo (bias).

Dado:
- `w = [0.5, -0.3, 0.8, 0.1, -0.6]` (pesos)
- `x = [2.0, 1.5, 0.8, 3.2, 0.5]` (entrada)
- `b = 0.2` (sesgo)

1. Calcula la salida de la neurona
2. Aplica la función de activación ReLU: `relu(z) = max(0, z)`
3. Aplica la función sigmoid: `sigmoid(z) = 1 / (1 + e^-z)`
4. ¿Qué pasa si duplicas todos los pesos? ¿Y si los pones a cero?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np

w = np.array([0.5, -0.3, 0.8, 0.1, -0.6])
x = np.array([2.0, 1.5, 0.8, 3.2, 0.5])
b = 0.2

# 1. Salida
z = np.dot(w, x) + b
print(f"w · x = {np.dot(w, x):.4f}")
print(f"z = w · x + b = {z:.4f}")

# 2. ReLU
relu = max(0, z)
print(f"\nReLU(z) = max(0, {z:.4f}) = {relu:.4f}")

# 3. Sigmoid
sigmoid = 1 / (1 + np.exp(-z))
print(f"sigmoid(z) = 1/(1+e^-{z:.2f}) = {sigmoid:.4f}")

# 4. Variaciones
print(f"\nPesos × 2: z = {np.dot(2*w, x) + b:.4f}")
print(f"Pesos × 0: z = {np.dot(0*w, x) + b:.4f}  (solo el bias)")
```
</details>

---
### Ejercicio 3 — Distancia entre puntos

**Enunciado:** La distancia euclidiana entre dos puntos es la magnitud del vector diferencia.

```
distancia(v, u) = ‖v - u‖
```

Tienes los vectores de perfil de cinco clientes (5 dimensiones: edad, ingresos, compras_mes, años_cliente, satisfaccion):
```
cliente_1 = [35, 45000, 8,  3, 4.2]
cliente_2 = [37, 48000, 9,  4, 4.5]
cliente_3 = [28, 28000, 3,  1, 3.1]
cliente_4 = [55, 80000, 15, 8, 4.8]
cliente_5 = [36, 46000, 8,  3, 4.3]
```

1. ¿Qué par de clientes es más similar según distancia euclidiana?
2. ¿Y según similitud coseno?
3. Normaliza los vectores (resta la media, divide por la desviación estándar) antes de calcular. ¿Cambia el resultado?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
from itertools import combinations

clientes = {
    "C1": np.array([35, 45000, 8,  3, 4.2]),
    "C2": np.array([37, 48000, 9,  4, 4.5]),
    "C3": np.array([28, 28000, 3,  1, 3.1]),
    "C4": np.array([55, 80000, 15, 8, 4.8]),
    "C5": np.array([36, 46000, 8,  3, 4.3]),
}

nombres = list(clientes.keys())
vectores = list(clientes.values())

# 1. Distancia euclidiana
print("Distancias euclidianas:")
min_dist, par_dist = float('inf'), None
for n1, n2 in combinations(nombres, 2):
    dist = np.linalg.norm(clientes[n1] - clientes[n2])
    print(f"  {n1} ↔ {n2}: {dist:.2f}")
    if dist < min_dist:
        min_dist, par_dist = dist, (n1, n2)
print(f"\nPar más similar (euclidiana): {par_dist} ({min_dist:.2f})")

# 2. Similitud coseno
print("\nSimilitudes coseno:")
max_sim, par_cos = -1, None
for n1, n2 in combinations(nombres, 2):
    v1, v2 = clientes[n1], clientes[n2]
    sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    print(f"  {n1} ↔ {n2}: {sim:.6f}")
    if sim > max_sim:
        max_sim, par_cos = sim, (n1, n2)
print(f"\nPar más similar (coseno): {par_cos} ({max_sim:.6f})")

# 3. Normalizar (z-score)
matriz = np.vstack(vectores)
media   = matriz.mean(axis=0)
std     = matriz.std(axis=0)
norm    = (matriz - media) / std
clientes_norm = {n: norm[i] for i, n in enumerate(nombres)}

print("\nDistancias con normalización z-score:")
min_dist_n, par_dist_n = float('inf'), None
for n1, n2 in combinations(nombres, 2):
    dist = np.linalg.norm(clientes_norm[n1] - clientes_norm[n2])
    print(f"  {n1} ↔ {n2}: {dist:.4f}")
    if dist < min_dist_n:
        min_dist_n, par_dist_n = dist, (n1, n2)
print(f"\nPar más similar (normalizado): {par_dist_n}")
```
</details>

---
### Ejercicio 4 — Buscador semántico

**Enunciado:** Implementa una función `buscar(consulta, documentos)` que:

1. Recibe un vector de consulta y un diccionario de vectores de documentos
2. Calcula la similitud coseno entre la consulta y cada documento
3. Devuelve los documentos ordenados de más a menos similar

Pruébalo con estos vectores de noticias (dimensiones: tecnología, deportes, política, cultura, economía):
```
noticias = {
  "IA supera a humanos en ajedrez": [0.8, 0.3, 0.1, 0.2, 0.1],
  "El Barça gana la Champions":     [0.1, 0.9, 0.0, 0.2, 0.0],
  "Nuevas leyes de privacidad IA":  [0.7, 0.0, 0.8, 0.1, 0.3],
  "Festival de Cannes 2025":        [0.1, 0.1, 0.1, 0.9, 0.1],
  "Inversión récord en startups":   [0.5, 0.0, 0.2, 0.1, 0.8],
  "Robot aprende a caminar solo":   [0.9, 0.2, 0.0, 0.1, 0.2],
}
consulta = [0.9, 0.1, 0.1, 0.0, 0.1]  # búsqueda sobre tecnología
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np

def similitud_coseno(v, u):
    v, u = np.array(v), np.array(u)
    return np.dot(v, u) / (np.linalg.norm(v) * np.linalg.norm(u))

def buscar(consulta, documentos, top_n=None):
    similitudes = [
        (titulo, similitud_coseno(consulta, vec))
        for titulo, vec in documentos.items()
    ]
    similitudes.sort(key=lambda x: x[1], reverse=True)
    return similitudes[:top_n] if top_n else similitudes

noticias = {
    "IA supera a humanos en ajedrez": [0.8, 0.3, 0.1, 0.2, 0.1],
    "El Barça gana la Champions":     [0.1, 0.9, 0.0, 0.2, 0.0],
    "Nuevas leyes de privacidad IA":  [0.7, 0.0, 0.8, 0.1, 0.3],
    "Festival de Cannes 2025":        [0.1, 0.1, 0.1, 0.9, 0.1],
    "Inversión récord en startups":   [0.5, 0.0, 0.2, 0.1, 0.8],
    "Robot aprende a caminar solo":   [0.9, 0.2, 0.0, 0.1, 0.2],
}
consulta = np.array([0.9, 0.1, 0.1, 0.0, 0.1])

resultados = buscar(consulta, noticias)
print("Resultados para consulta de tecnología:")
print(f"{'Pos':<4} {'Similitud':>10}  Noticia")
for i, (titulo, sim) in enumerate(resultados, 1):
    barra = "█" * int(sim * 20)
    print(f"  {i}.  {sim:.4f}  {barra:<20}  {titulo}")

# Probar otra consulta
consulta2 = np.array([0.0, 0.9, 0.0, 0.0, 0.0])  # deportes
print("\nResultados para consulta de deportes:")
for titulo, sim in buscar(consulta2, noticias):
    print(f"  {sim:.4f}  {titulo}")
```
</details>

---
### Ejercicio 5 (reto) — Sistema de recomendación

**Enunciado:** Construye un sistema de recomendación de películas basado en similitud coseno.

Cada película está representada por un vector de géneros (acción, comedia, drama, terror, ciencia_ficción, romance):

```python
peliculas = {
    "Interstellar":     [0.3, 0.1, 0.6, 0.0, 0.9, 0.1],
    "The Dark Knight":  [0.9, 0.1, 0.7, 0.2, 0.3, 0.0],
    "Toy Story":        [0.2, 0.9, 0.4, 0.0, 0.3, 0.2],
    "El Conjuro":       [0.1, 0.0, 0.4, 0.9, 0.1, 0.0],
    "La La Land":       [0.1, 0.3, 0.8, 0.0, 0.0, 0.9],
    "Avengers":         [0.9, 0.4, 0.3, 0.1, 0.8, 0.1],
    "Hereditary":       [0.0, 0.0, 0.7, 0.9, 0.2, 0.0],
    "Her":              [0.0, 0.2, 0.8, 0.0, 0.7, 0.6],
    "Superbad":         [0.1, 0.9, 0.3, 0.0, 0.0, 0.2],
    "Arrival":          [0.1, 0.0, 0.8, 0.1, 0.9, 0.3],
}
```

Implementa:
1. `recomendar(pelicula_vista, n=3)` — recomienda las N más similares
2. `perfil_usuario(peliculas_vistas)` — crea el vector perfil del usuario como la media de lo que ha visto
3. `recomendar_para_usuario(perfil, no_vistas, n=3)` — recomienda según el perfil

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np

peliculas = {
    "Interstellar":    [0.3, 0.1, 0.6, 0.0, 0.9, 0.1],
    "The Dark Knight": [0.9, 0.1, 0.7, 0.2, 0.3, 0.0],
    "Toy Story":       [0.2, 0.9, 0.4, 0.0, 0.3, 0.2],
    "El Conjuro":      [0.1, 0.0, 0.4, 0.9, 0.1, 0.0],
    "La La Land":      [0.1, 0.3, 0.8, 0.0, 0.0, 0.9],
    "Avengers":        [0.9, 0.4, 0.3, 0.1, 0.8, 0.1],
    "Hereditary":      [0.0, 0.0, 0.7, 0.9, 0.2, 0.0],
    "Her":             [0.0, 0.2, 0.8, 0.0, 0.7, 0.6],
    "Superbad":        [0.1, 0.9, 0.3, 0.0, 0.0, 0.2],
    "Arrival":         [0.1, 0.0, 0.8, 0.1, 0.9, 0.3],
}
peliculas = {k: np.array(v) for k, v in peliculas.items()}
GENEROS = ["acción", "comedia", "drama", "terror", "sci-fi", "romance"]

def similitud_coseno(v, u):
    return np.dot(v, u) / (np.linalg.norm(v) * np.linalg.norm(u))

def recomendar(pelicula_vista, n=3):
    base = peliculas[pelicula_vista]
    similitudes = [
        (titulo, similitud_coseno(base, vec))
        for titulo, vec in peliculas.items()
        if titulo != pelicula_vista
    ]
    return sorted(similitudes, key=lambda x: x[1], reverse=True)[:n]

def perfil_usuario(peliculas_vistas):
    vectores = [peliculas[p] for p in peliculas_vistas]
    return np.mean(vectores, axis=0)

def recomendar_para_usuario(peliculas_vistas, n=3):
    perfil = perfil_usuario(peliculas_vistas)
    print(f"Perfil del usuario (géneros más afines):")
    for g, v in sorted(zip(GENEROS, perfil), key=lambda x: -x[1]):
        print(f"  {g}: {v:.2f}")
    candidatas = {t: v for t, v in peliculas.items() if t not in peliculas_vistas}
    similitudes = [(t, similitud_coseno(perfil, v)) for t, v in candidatas.items()]
    return sorted(similitudes, key=lambda x: x[1], reverse=True)[:n]

print("Si te gustó 'Interstellar', te recomendamos:")
for titulo, sim in recomendar("Interstellar"):
    print(f"  {sim:.4f}  {titulo}")

vistas = ["Interstellar", "Arrival", "Her"]
print(f"\nUsuario ha visto: {vistas}")
recomendaciones = recomendar_para_usuario(vistas)
print("\nRecomendaciones:")
for titulo, sim in recomendaciones:
    print(f"  {sim:.4f}  {titulo}")
```
</details>

---
## Resumen

```python
import numpy as np

v = np.array([3, 4])
u = np.array([1, 2])

# Magnitud
np.linalg.norm(v)          # 5.0

# Vector unitario (dirección)
v / np.linalg.norm(v)      # [0.6, 0.8]

# Producto punto
np.dot(v, u)   # o  v @ u

# Similitud coseno
np.dot(v, u) / (np.linalg.norm(v) * np.linalg.norm(u))

# Ángulo entre vectores
theta = np.degrees(np.arccos(np.clip(similitud, -1, 1)))
```

## ¿Qué viene en la próxima clase?
En la **Clase 2** pasamos a matrices: tablas de números que representan transformaciones y son el motor de las redes neuronales.